In [4]:
import gmsh
import numpy as np
from mpi4py import MPI
from dolfinx.io import XDMFFile, gmshio
from dolfinx.io.gmshio import read_from_msh

In [5]:

def draw_mesh(r1, r2, angle):
    gmsh.initialize()
    gmsh.clear()
    gmsh.model.add("test")
    gmsh.model.setCurrent("test")

    gmsh.model.occ.addPoint(250e-3,0,0,meshSize=10e-3, tag=1)
    gmsh.model.occ.addPoint(250e-3, 500e-3, 0, meshSize=10e-3, tag=2)
    gmsh.model.occ.addPoint(0,0,0,meshSize=10e-3, tag=3)
    gmsh.model.occ.addPoint(0,500e-3,0, meshSize=10e-3, tag=4)

    gmsh.model.occ.addLine(3,1,tag=1)
    gmsh.model.occ.addLine(3,4,tag=2)
    gmsh.model.occ.addLine(1,2,tag=3)
    gmsh.model.occ.addLine(2,4,tag=4)

    gmsh.model.occ.addCurveLoop([3,4,-2,1], tag=1)


    
    gmsh.model.occ.addPoint(80e-3+r1, 150e-3, 0, meshSize=4e-3, tag=20)   
    gmsh.model.occ.addEllipse(80e-3,150e-3,0, r1,r2, tag=5, angle1=0, angle2=2*np.pi,) 
    gmsh.model.occ.rotate([(0,20),(1,5)],80e-3,150e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([5], tag=2)

    gmsh.model.occ.addPoint(170e-3+r1, 150e-3, 0, meshSize=4e-3, tag=22)
    gmsh.model.occ.addEllipse(170e-3,150e-3,0, r1,r2, tag=6, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,22),(1,6)],170e-3,150e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([6], tag=3)

    gmsh.model.occ.addPoint(80e-3+r1, 250e-3, 0, meshSize=4e-3, tag=24)
    gmsh.model.occ.addEllipse(80e-3,250e-3,0, r1,r2, tag=7, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,24),(1,7)],80e-3,250e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([7], tag=4)

    
    gmsh.model.occ.addPoint(170e-3+r1, 250e-3, 0, meshSize=4e-3, tag=26)
    gmsh.model.occ.addEllipse(170e-3,250e-3,0, r1,r2, tag=8, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,26),(1,8)],170e-3,250e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([8], tag=5)

    gmsh.model.occ.addPoint(80e-3+r1, 350e-3, 0, meshSize=4e-3, tag=28)
    gmsh.model.occ.addEllipse(80e-3,350e-3,0, r1,r2, tag=9, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,28),(1,9)],80e-3,350e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([9], tag=6)

    gmsh.model.occ.addPoint(170e-3+r1, 350e-3, 0, meshSize=4e-3, tag=30)
    gmsh.model.occ.addEllipse(170e-3,350e-3,0, r1,r2, tag=10, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,30),(1,10)],170e-3,350e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([10], tag=7)


    gmsh.model.occ.addPlaneSurface([1,2,3,4,5,6,7], tag=1)


    gmsh.model.occ.synchronize()

    gmsh.model.addPhysicalGroup(1, [1], 1, "fixed")
    gmsh.model.addPhysicalGroup(1, [4], 2, "load")
    gmsh.model.addPhysicalGroup(1, [2,3,5,6,7,8,9,10], 3, "free")
    gmsh.model.addPhysicalGroup(2, [1], 4, "domain")

    gmsh.model.mesh.generate(2)
    gmsh.model.mesh.optimize("Netgen")

    model_rank = 0
    mesh_comm = MPI.COMM_WORLD
    domain, ct, ft = gmshio.model_to_mesh(gmsh.model, mesh_comm, model_rank, gdim=2)
    gmsh.finalize()
    return domain

In [3]:
"""
import pyvista
from dolfinx.plot import vtk_mesh

domain, ct=draw_mesh(40e-3, 10e-3, 30)
pyvista.start_xvfb()
plotter = pyvista.Plotter()
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, tdim)
grid = pyvista.UnstructuredGrid(*vtk_mesh(domain, tdim))
num_local_cells = domain.topology.index_map(tdim).size_local
grid.cell_data["Marker"] = ct.values[ct.indices < num_local_cells]
grid.set_active_scalars("Marker")
actor = plotter.add_mesh(grid, show_edges=True, color="white")
plotter.view_xy()

plotter.show()
"""

'\nimport pyvista\nfrom dolfinx.plot import vtk_mesh\n\ndomain, ct=draw_mesh(40e-3, 10e-3, 30)\npyvista.start_xvfb()\nplotter = pyvista.Plotter()\ntdim = domain.topology.dim\ndomain.topology.create_connectivity(tdim, tdim)\ngrid = pyvista.UnstructuredGrid(*vtk_mesh(domain, tdim))\nnum_local_cells = domain.topology.index_map(tdim).size_local\ngrid.cell_data["Marker"] = ct.values[ct.indices < num_local_cells]\ngrid.set_active_scalars("Marker")\nactor = plotter.add_mesh(grid, show_edges=True, color="white")\nplotter.view_xy()\n\nplotter.show()\n'

In [ ]:
import os
from dolfinx import mesh, fem, plot, io, default_scalar_type
from dolfinx.fem.petsc import LinearProblem
from mpi4py import MPI
import ufl
import numpy as np
from dolfinx import geometry




def epsilon(u):
    return ufl.sym(ufl.grad(u))  # Equivalent to 0.5*(ufl.nabla_grad(u) + ufl.nabla_grad(u).T)


def sigma(u):
    return lambda_ * ufl.nabla_div(u) * ufl.Identity(len(u)) + 2 * mu * epsilon(u)



if not os.path.exists("data_angle"):
    os.makedirs("data_angle")

N=1000
r1=(25e-3)*np.ones(1000)
r2=(25e-3)*np.ones(1000)
theta=np.arange(40, 140, 100/1000)
angle=np.zeros(1000)
for n in range(N):
    
    
    domain=draw_mesh(r1[n],r2[n], angle[n])
    E = fem.Constant(domain, 210e9)
    nu = fem.Constant(domain, 0.3)

    lmbda = E * nu / (1 + nu) / (1 - 2 * nu)
    mu = E / 2 / (1 + nu)

    beta = 1.25
    lambda_ = beta

    V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim, )))

    def clamped_boundary(x):
        return np.isclose(x[1], 0)


    fdim = domain.topology.dim - 1
    boundary_facets = mesh.locate_entities_boundary(domain, fdim, clamped_boundary)

    u_D = np.array([0, 0], dtype=default_scalar_type)
    bc = fem.dirichletbc(u_D, fem.locate_dofs_topological(V, fdim, boundary_facets), V)

    boundaries = [(1, lambda x: np.isclose(x[0], 0)),
                (2, lambda x: np.isclose(x[0], 250e-3)),
                (3, lambda x: np.isclose(x[1], 500e-3))]

    facet_indices, facet_markers = [], []
    fdim = domain.topology.dim - 1
    for (marker, locator) in boundaries:
        facets = mesh.locate_entities(domain, fdim, locator)
        facet_indices.append(facets)
        facet_markers.append(np.full_like(facets, marker))
    facet_indices = np.hstack(facet_indices).astype(np.int32)
    facet_markers = np.hstack(facet_markers).astype(np.int32)
    sorted_facets = np.argsort(facet_indices)
    facet_tag = mesh.meshtags(domain, fdim, facet_indices[sorted_facets], facet_markers[sorted_facets])
    ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tag)
    T0 = fem.Constant(domain, default_scalar_type((0, 0)))
    T1 = fem.Constant(domain, default_scalar_type((100e6*np.cos(theta[n]*np.pi/180), 100e6*np.sin(theta[n]*np.pi/180))))
    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    f = fem.Constant(domain, default_scalar_type((0, 0)))
    a = ufl.inner(sigma(u), epsilon(v)) * ufl.dx
    L = ufl.dot(f, v) * ufl.dx + ufl.dot(T0, v) * ds(1) + ufl.dot(T0, v) * ds(2) + + ufl.dot(T1, v) * ds(3)
    problem = LinearProblem(a, L, bcs=[bc], petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    uh = problem.solve()
    
    s = sigma(uh) - 1. / 3 * ufl.tr(sigma(uh)) * ufl.Identity(len(uh))
    von_Mises = ufl.sqrt(3. / 2 * ufl.inner(s, s))

    V_von_mises = fem.functionspace(domain, ("DG", 0))
    stress_expr = fem.Expression(von_Mises, V_von_mises.element.interpolation_points())
    stresses = fem.Function(V_von_mises)
    stresses.interpolate(stress_expr)

    cell_to_vertices = domain.topology.connectivity(domain.topology.dim, 0).array.reshape(-1,3)
    points= domain.geometry.x.reshape(-1,3)

    bb_tree = geometry.bb_tree(domain, domain.topology.dim)
    cells = []
    points_on_proc = []
    # Find cells whose bounding-box collide with the the points
    cell_candidates = geometry.compute_collisions_points(bb_tree, points)
    # Choose one of the cells that contains the point
    colliding_cells = geometry.compute_colliding_cells(domain, cell_candidates, points)
    for i, point in enumerate(points):
        if len(colliding_cells.links(i)) > 0:
            points_on_proc.append(point)
            cells.append(colliding_cells.links(i)[0])
    val=stresses.eval(domain.geometry.x, cells)

        
    #if not os.path.exists("data_angle/"+str(n)):
    #    os.makedirs("data_angle/"+str(n))
    #np.save("data_angle/"+str(n)+"/H_holes.npy", np.array([[80e-3,150e-3, r1[n],r2[n],angle[n]],[170e-3,150e-3,r1[n],r2[n],angle[n]],[80e-3, 250e-3, r1[n],r2[n],angle[n]],[170e-3, 250e-3, r1[n],r2[n],angle[n]],[80e-3, 350e-3, r1[n],r2[n],angle[n]],[170e-3, 350e-3, r1[n],r2[n],angle[n]]]))
    #np.save("data_angle/"+str(n)+"/H_angle.npy", np.array([theta[n]]))
    #np.save("data_angle/"+str(n)+"/H_mesh_geometry.npy", points[:,:2])
    #np.save("data_angle/"+str(n)+"/H_mesh_topology.npy", cell_to_vertices)
    #np.save("data_angle/"+str(n)+"/H_y.npy", val)

    
   


        

    
    


Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 50%] Meshing curve 5 (Ellipse)
Info    : [ 60%] Meshing curve 6 (Ellipse)
Info    : [ 70%] Meshing curve 7 (Ellipse)
Info    : [ 80%] Meshing curve 8 (Ellipse)
Info    : [ 90%] Meshing curve 9 (Ellipse)
Info    : [100%] Meshing curve 10 (Ellipse)
Info    : Done meshing 1D (Wall 0.000961417s, CPU 0.001299s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.146347s, CPU 0.146864s)
Info    : 4059 nodes 8132 elements
Info    : Optimizing mesh (Netgen)...
Info    : Done optimizing mesh (Wall 1.2e-06s, CPU 2e-06s)
Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing cur

In [5]:
#x,y=np.meshgrid(np.array([0,2,4]),np.array([1,3,5,7,9]))
#x.reshape(-1).reshape(5,3)

In [6]:
#import matplotlib.pyplot as plt
#plt.contourf(x, y, final_val.reshape(500,250), 800)

In [4]:
def draw_mesh(r1, r2,angle):
    gmsh.initialize()
    gmsh.clear()
    gmsh.model.add("test")
    gmsh.model.setCurrent("test")

    gmsh.model.occ.addPoint(250e-3,0,0,meshSize=40e-3, tag=1)
    gmsh.model.occ.addPoint(250e-3, 500e-3, 0, meshSize=40e-3, tag=2)
    gmsh.model.occ.addPoint(0,0,0,meshSize=40e-3, tag=3)
    gmsh.model.occ.addPoint(0,500e-3,0, meshSize=40e-3, tag=4)

    gmsh.model.occ.addLine(3,1,tag=1)
    gmsh.model.occ.addLine(3,4,tag=2)
    gmsh.model.occ.addLine(1,2,tag=3)
    gmsh.model.occ.addLine(2,4,tag=4)

    gmsh.model.occ.addCurveLoop([3,4,-2,1], tag=1)


    
    gmsh.model.occ.addPoint(80e-3+r1, 150e-3, 0, meshSize=16e-3, tag=20)   
    gmsh.model.occ.addEllipse(80e-3,150e-3,0, r1,r2, tag=5, angle1=0, angle2=2*np.pi,) 
    gmsh.model.occ.rotate([(0,20),(1,5)],80e-3,150e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([5], tag=2)

    gmsh.model.occ.addPoint(170e-3+r1, 150e-3, 0, meshSize=16e-3, tag=22)
    gmsh.model.occ.addEllipse(170e-3,150e-3,0, r1,r2, tag=6, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,22),(1,6)],170e-3,150e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([6], tag=3)

    gmsh.model.occ.addPoint(80e-3+r1, 250e-3, 0, meshSize=16e-3, tag=24)
    gmsh.model.occ.addEllipse(80e-3,250e-3,0, r1,r2, tag=7, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,24),(1,7)],80e-3,250e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([7], tag=4)

    
    gmsh.model.occ.addPoint(170e-3+r1, 250e-3, 0, meshSize=16e-3, tag=26)
    gmsh.model.occ.addEllipse(170e-3,250e-3,0, r1,r2, tag=8, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,26),(1,8)],170e-3,250e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([8], tag=5)

    gmsh.model.occ.addPoint(80e-3+r1, 350e-3, 0, meshSize=16e-3, tag=28)
    gmsh.model.occ.addEllipse(80e-3,350e-3,0, r1,r2, tag=9, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,28),(1,9)],80e-3,350e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([9], tag=6)

    gmsh.model.occ.addPoint(170e-3+r1, 350e-3, 0, meshSize=16e-3, tag=30)
    gmsh.model.occ.addEllipse(170e-3,350e-3,0, r1,r2, tag=10, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,30),(1,10)],170e-3,350e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([10], tag=7)


    gmsh.model.occ.addPlaneSurface([1,2,3,4,5,6,7], tag=1)


    gmsh.model.occ.synchronize()

    gmsh.model.addPhysicalGroup(1, [1], 1, "fixed")
    gmsh.model.addPhysicalGroup(1, [4], 2, "load")
    gmsh.model.addPhysicalGroup(1, [2,3,5,6,7,8,9,10], 3, "free")
    gmsh.model.addPhysicalGroup(2, [1], 4, "domain")

    gmsh.model.mesh.generate(2)
    gmsh.model.mesh.optimize("Netgen")

    model_rank = 0
    mesh_comm = MPI.COMM_WORLD
    domain, ct, ft = gmshio.model_to_mesh(gmsh.model, mesh_comm, model_rank, gdim=2)
    gmsh.finalize()
    return domain
    
    

In [8]:
"""
pyvista.start_xvfb()
plotter = pyvista.Plotter()
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, tdim)
grid = pyvista.UnstructuredGrid(*vtk_mesh(domain, tdim))
num_local_cells = domain.topology.index_map(tdim).size_local
grid.cell_data["Marker"] = ct.values[ct.indices < num_local_cells]
grid.set_active_scalars("Marker")
actor = plotter.add_mesh(grid, show_edges=True, color="white")
plotter.view_xy()

plotter.show()
"""

'\npyvista.start_xvfb()\nplotter = pyvista.Plotter()\ntdim = domain.topology.dim\ndomain.topology.create_connectivity(tdim, tdim)\ngrid = pyvista.UnstructuredGrid(*vtk_mesh(domain, tdim))\nnum_local_cells = domain.topology.index_map(tdim).size_local\ngrid.cell_data["Marker"] = ct.values[ct.indices < num_local_cells]\ngrid.set_active_scalars("Marker")\nactor = plotter.add_mesh(grid, show_edges=True, color="white")\nplotter.view_xy()\n\nplotter.show()\n'

In [5]:


def epsilon(u):
    return ufl.sym(ufl.grad(u))  # Equivalent to 0.5*(ufl.nabla_grad(u) + ufl.nabla_grad(u).T)


def sigma(u):
    return lambda_ * ufl.nabla_div(u) * ufl.Identity(len(u)) + 2 * mu * epsilon(u)



if not os.path.exists("data_angle"):
    os.makedirs("data_angle")


for n in range(N):
    domain=draw_mesh(r1[n],r2[n], angle[n])
    E = fem.Constant(domain, 210e9)
    nu = fem.Constant(domain, 0.3)

    lmbda = E * nu / (1 + nu) / (1 - 2 * nu)
    mu = E / 2 / (1 + nu)

    beta = 1.25
    lambda_ = beta

    V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim, )))

    def clamped_boundary(x):
        return np.isclose(x[1], 0)


    fdim = domain.topology.dim - 1
    boundary_facets = mesh.locate_entities_boundary(domain, fdim, clamped_boundary)

    u_D = np.array([0, 0], dtype=default_scalar_type)
    bc = fem.dirichletbc(u_D, fem.locate_dofs_topological(V, fdim, boundary_facets), V)

    boundaries = [(1, lambda x: np.isclose(x[0], 0)),
                (2, lambda x: np.isclose(x[0], 250e-3)),
                (3, lambda x: np.isclose(x[1], 500e-3))]

    facet_indices, facet_markers = [], []
    fdim = domain.topology.dim - 1
    for (marker, locator) in boundaries:
        facets = mesh.locate_entities(domain, fdim, locator)
        facet_indices.append(facets)
        facet_markers.append(np.full_like(facets, marker))
    facet_indices = np.hstack(facet_indices).astype(np.int32)
    facet_markers = np.hstack(facet_markers).astype(np.int32)
    sorted_facets = np.argsort(facet_indices)
    facet_tag = mesh.meshtags(domain, fdim, facet_indices[sorted_facets], facet_markers[sorted_facets])
    ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tag)
    
    
    
    T0 = fem.Constant(domain, default_scalar_type((0, 0)))
    T1 = fem.Constant(domain, default_scalar_type((100e6*np.cos(theta[n]*np.pi/180), 100e6*np.sin(theta[n]*np.pi/180))))
    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    f = fem.Constant(domain, default_scalar_type((0, 0)))
    a = ufl.inner(sigma(u), epsilon(v)) * ufl.dx
    L = ufl.dot(f, v) * ufl.dx + ufl.dot(T0, v) * ds(1) + ufl.dot(T0, v) * ds(2) + + ufl.dot(T1, v) * ds(3)
    problem = LinearProblem(a, L, bcs=[bc], petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    uh = problem.solve()
    
    s = sigma(uh) - 1. / 3 * ufl.tr(sigma(uh)) * ufl.Identity(len(uh))
    von_Mises = ufl.sqrt(3. / 2 * ufl.inner(s, s))

    V_von_mises = fem.functionspace(domain, ("DG", 0))
    stress_expr = fem.Expression(von_Mises, V_von_mises.element.interpolation_points())
    stresses = fem.Function(V_von_mises)
    stresses.interpolate(stress_expr)

    cell_to_vertices = domain.topology.connectivity(domain.topology.dim, 0).array.reshape(-1,3)
    points= domain.geometry.x.reshape(-1,3)

    bb_tree = geometry.bb_tree(domain, domain.topology.dim)
    cells = []
    points_on_proc = []
    # Find cells whose bounding-box collide with the the points
    cell_candidates = geometry.compute_collisions_points(bb_tree, points)
    # Choose one of the cells that contains the point
    colliding_cells = geometry.compute_colliding_cells(domain, cell_candidates, points)
    for i, point in enumerate(points):
        if len(colliding_cells.links(i)) > 0:
            points_on_proc.append(point)
            cells.append(colliding_cells.links(i)[0])
    val=stresses.eval(domain.geometry.x, cells)

    
    
    if not os.path.exists("data_angle/"+str(n)):
        os.makedirs("data_angle/"+str(n))
    np.save("data_angle/"+str(n)+"/L_holes.npy", np.array([[80e-3,150e-3, r1[n],r2[n],angle[n]],[170e-3,150e-3,r1[n],r2[n],angle[n]],[80e-3, 250e-3, r1[n],r2[n],angle[n]],[170e-3, 250e-3, r1[n],r2[n],angle[n]],[80e-3, 350e-3, r1[n],r2[n],angle[n]],[170e-3, 350e-3, r1[n],r2[n],angle[n]]]))
    np.save("data_angle/"+str(n)+"/L_angle.npy", np.array([theta[n]]))
    np.save("data_angle/"+str(n)+"/L_mesh_geometry.npy", points[:,:2])
    np.save("data_angle/"+str(n)+"/L_mesh_topology.npy", cell_to_vertices)
    np.save("data_angle/"+str(n)+"/L_y.npy", val)
    

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 50%] Meshing curve 5 (Ellipse)
Info    : [ 60%] Meshing curve 6 (Ellipse)
Info    : [ 70%] Meshing curve 7 (Ellipse)
Info    : [ 80%] Meshing curve 8 (Ellipse)
Info    : [ 90%] Meshing curve 9 (Ellipse)
Info    : [100%] Meshing curve 10 (Ellipse)
Info    : Done meshing 1D (Wall 0.000925311s, CPU 0.001298s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0275932s, CPU 0.027882s)
Info    : 339 nodes 692 elements
Info    : Optimizing mesh (Netgen)...
Info    : Done optimizing mesh (Wall 1.61e-05s, CPU 1.6e-05s)
Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing c

In [10]:
"""
pyvista.start_xvfb()

# Create plotter and pyvista grid
p = pyvista.Plotter()
topology, cell_types, geo = plot.vtk_mesh(V)
grid = pyvista.UnstructuredGrid(topology, cell_types, geo)

# Attach vector values to grid and warp grid by vector
grid["VonMises"] = stresses.x.array
grid.set_active_scalars("VonMises")
p.add_mesh(grid, show_edges=False)

p.show_axes()

p.show()
"""

'\npyvista.start_xvfb()\n\n# Create plotter and pyvista grid\np = pyvista.Plotter()\ntopology, cell_types, geo = plot.vtk_mesh(V)\ngrid = pyvista.UnstructuredGrid(topology, cell_types, geo)\n\n# Attach vector values to grid and warp grid by vector\ngrid["VonMises"] = stresses.x.array\ngrid.set_active_scalars("VonMises")\np.add_mesh(grid, show_edges=False)\n\np.show_axes()\n\np.show()\n'